In [ ]:
!pip install sentence-transformers transformers faiss-cpu rank-bm25 torch

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.5/18.5 MB 81.0 MB/s eta 0:00:00


In [ ]:
import numpy as np
import faiss
import torch

from sentence_transformers import SentenceTransformer,CrossEncoder
from rank_bm25 import BM25Okapi

In [ ]:
documents = [
    "Employees are entitled to 20 days of paid annual leave each year.",
    "Work from home is permitted up to three days per week with manager approval.",
    "Medical insurance covers employees, spouses, and up to two children.",
    "Employees must submit travel expense claims within 30 days of the trip.",
    "The standard working hours are from 9:00 AM to 6:00 PM, Monday through Friday.",
    "Performance reviews are conducted twice a year in June and December.",
    "Employees receive maternity leave of 26 weeks and paternity leave of 15 days.",
    "Laptop replacement requests should be submitted through the IT helpdesk portal.",
    "The probation period for new employees is six months.",
    "Confidential company information must not be shared outside the organization."
]


In [ ]:
embedder=SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
embeddings=embedder.encode(documents)

In [ ]:
dimension=embeddings.shape[1]
index=faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings).astype('float32'))

In [ ]:
tokenized_docs=[doc.lower().split() for doc in documents]

bm25=BM25Okapi(tokenized_docs)

In [ ]:
reranker=CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 90.9MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/1.33k [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/711k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

In [ ]:
!pip install openai

In [ ]:
from openai import OpenAI

client=OpenAI(
    api_key="",
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
def dense_search(query,k=5):
  query_embeddings=embedder.encode([query]).astype("float32")
  distances,indices=index.search(query_embeddings,k)
  return [documents[i] for i in indices[0]]

In [ ]:
def bm25_search(query,k=5):
  scores=bm25.get_scores(query.lower().split())
  ranked=np.argsort(scores)[::-1][:k]
  return [documents[i] for i in ranked]

In [ ]:
def merge_results(vector_docs,bm25_docs):
  merged=[]
  for doc in vector_docs+bm25_docs:
    if doc not in merged:
      merged.append(doc)
  return merged

In [ ]:
def rerank(query,docs,top_k=3):
  pairs=[[query,doc] for doc in docs]
  scores=reranker.predict(pairs)
  ranked=sorted(
      zip(scores,docs),
      reverse=True
  )

  return [doc for score,doc in ranked[:top_k]]

In [ ]:
def generate_answer(query, context):

    prompt = f"""
You are a helpful assistant.

Context:
{context}

Question:
{query}

Answer using only the provided context.
"""

    response = client.chat.completions.create(
        model="llama-3.3-70b-versatile",      # or any Groq model you have access to
        messages=[
            {
                "role": "user",
                "content": prompt
            }
        ],
        temperature=0
    )

    return response.choices[0].message.content


In [ ]:
def hybrid_rag(query):
  print("Query:",query)
  print("-----------------------------------")

  dense_docs=dense_search(query)
  print("\nDense Retrieval:")
  for d in dense_docs:
    print("-",d)

  bm25_docs=bm25_search(query)
  print("\nBM25 retrival:")
  for d in bm25_docs:
    print("-",d)

  merged_docs=merge_results(dense_docs,bm25_docs)
  print("\nMerged Results:")
  for d in merged_docs:
    print("-",d)

  final_docs=rerank(query,merged_docs)
  print("\nFinal Results:")
  for d in final_docs:
    print("-",d)

  context="\n".join(final_docs)

  answer=generate_answer(query,context)

  return final_docs,answer

In [ ]:
def normal_rag(query):
  docs=dense_search(query)
  context="\n".join(docs)
  answer=generate_answer(query,context)
  return docs,answer

In [ ]:
query="Tell me about the leave policies"

normal_docs,normal_answer=normal_rag(query)

print("----------------------")
print("normal rag :")

for d in normal_docs:
  print("-",d)

print("----------------------")
print("answer :")
print(normal_answer)

hybrid_docs,hybrid_answer=hybrid_rag(query)

print("----------------------")
print("hybrid rag :")

for d in hybrid_docs:
  print("-",d)

print("----------------------")
print("answer :")
print(hybrid_answer)

----------------------
normal rag :
- Employees are entitled to 20 days of paid annual leave each year.
- Employees receive maternity leave of 26 weeks and paternity leave of 15 days.
- Work from home is permitted up to three days per week with manager approval.
- The probation period for new employees is six months.
- Medical insurance covers employees, spouses, and up to two children.
----------------------
answer :
The company offers the following leave policies: 

1. Paid annual leave: 20 days per year.
2. Maternity leave: 26 weeks.
3. Paternity leave: 15 days.
Query: Tell me about the leave policies
-----------------------------------

Dense Retrieval:
- Employees are entitled to 20 days of paid annual leave each year.
- Employees receive maternity leave of 26 weeks and paternity leave of 15 days.
- Work from home is permitted up to three days per week with manager approval.
- The probation period for new employees is six months.
- Medical insurance covers employees, spouses, and 